# Exam (Hard) — Practice Problems

Stretch problems beyond likely screening difficulty. Still practical. Each problem has 3-4 interacting components with subtle edge cases.

**Mix:** 50% data pipelines / 50% ML

**Rules:**
- 30-minute timer per problem
- No docs, no autocomplete, no outside help
- Run the test cell — all assertions must pass
- Do MEDIUM first. Come here when those feel comfortable.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
import numpy as np
from typing import Any, Iterator
from collections import defaultdict
import math

---

## Problem 1: Near-Duplicate Detection Pipeline

### Scenario

Before training, you need to remove near-duplicate images from your dataset. You're given pre-computed embeddings for each image.

### Spec

**Cosine similarity:** `cos(a, b) = (a · b) / (||a|| * ||b||)`

Two items are **near-duplicates** if their cosine similarity >= `threshold`.

A **duplicate group** is a set of items where every item is a near-duplicate of at least one other item in the group (connected components — use union-find or BFS, your choice).

### Requirements

**`find_duplicate_pairs(embeddings: torch.Tensor, threshold: float) -> list[tuple[int, int]]`:**
- embeddings: (N, D)
- Return list of (i, j) pairs where i < j and cos_sim(i, j) >= threshold
- Use matrix operations, not nested loops

**`group_duplicates(num_items: int, pairs: list[tuple[int, int]]) -> list[set[int]]`:**
- Given duplicate pairs, find connected components
- Return only groups with 2+ items (singletons are unique, don't include them)

**`deduplicate(embeddings: torch.Tensor, quality_scores: torch.Tensor, threshold: float) -> dict`:**
- End-to-end: find pairs → group → for each group, keep the item with highest quality_score
- Return `{"kept": sorted list of kept indices, "removed": sorted list of removed indices, "groups": list of groups, "removal_rate": float}`

In [ ]:
def find_duplicate_pairs(embeddings: torch.Tensor, threshold: float) -> list[tuple[int, int]]:
    """Find all near-duplicate pairs by cosine similarity."""
    # YOUR CODE HERE
    pass


def group_duplicates(num_items: int, pairs: list[tuple[int, int]]) -> list[set[int]]:
    """Find connected components from duplicate pairs."""
    # YOUR CODE HERE
    pass


def deduplicate(embeddings: torch.Tensor, quality_scores: torch.Tensor, threshold: float) -> dict:
    """End-to-end deduplication pipeline."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 1 ---
torch.manual_seed(42)

# Create embeddings with known structure
base_a = F.normalize(torch.randn(1, 16), dim=1)
base_b = F.normalize(torch.randn(1, 16), dim=1)
unique = F.normalize(torch.randn(1, 16), dim=1)

embs = F.normalize(torch.cat([
    base_a + 0.01 * torch.randn(1, 16),  # 0 — group A
    base_a + 0.01 * torch.randn(1, 16),  # 1 — group A
    base_a + 0.01 * torch.randn(1, 16),  # 2 — group A
    base_b + 0.01 * torch.randn(1, 16),  # 3 — group B
    base_b + 0.01 * torch.randn(1, 16),  # 4 — group B
    unique,                                # 5 — unique
]), dim=1)
quality = torch.tensor([0.5, 0.9, 0.3, 0.7, 0.8, 0.6])

# find_duplicate_pairs
pairs = find_duplicate_pairs(embs, threshold=0.95)
assert all(i < j for i, j in pairs), "Pairs should have i < j"
# Items 0,1,2 should pair with each other; 3,4 should pair
pair_set = set(pairs)
assert (0, 1) in pair_set or (0, 2) in pair_set, "Group A pairs should be found"
assert (3, 4) in pair_set, "Group B pair should be found"
# Item 5 shouldn't pair with anyone
assert not any(5 in p for p in pairs), "Unique item shouldn't pair"

# group_duplicates
groups = group_duplicates(6, pairs)
assert len(groups) == 2, f"Expected 2 groups, got {len(groups)}"
all_grouped = set().union(*groups)
assert 5 not in all_grouped, "Unique item shouldn't be in any group"
group_a = [g for g in groups if 0 in g][0]
group_b = [g for g in groups if 3 in g][0]
assert group_a == {0, 1, 2}
assert group_b == {3, 4}

# deduplicate
result = deduplicate(embs, quality, threshold=0.95)
assert sorted(result["kept"]) == [1, 4, 5], f"Expected [1,4,5], got {sorted(result['kept'])}"
assert sorted(result["removed"]) == [0, 2, 3]
assert result["removal_rate"] == 3/6

# Edge: no duplicates
random_embs = F.normalize(torch.randn(5, 16), dim=1)
result_none = deduplicate(random_embs, torch.ones(5), threshold=0.99)
assert len(result_none["removed"]) == 0
assert len(result_none["kept"]) == 5

print("Problem 1: ALL TESTS PASSED")

---

## Problem 2: Bucketed DataLoader for Variable-Length Sequences

### Scenario

Video clips and captions have varying lengths. Naive batching wastes compute on padding. Build a bucketed loader that groups similar-length items together.

### Requirements

**`BucketedDataset(Dataset)`:**
- `__init__(self, records: list[dict], num_buckets: int)` — records have `"id"` (str), `"tokens"` (list[int]), `"label"` (int)
- Sort records by token length, divide into `num_buckets` equal-sized buckets
- `__getitem__` returns `{"id": str, "tokens": int64 tensor, "label": int64 scalar tensor}`
- `__len__` returns total records
- `get_bucket_boundaries(self) -> list[tuple[int, int]]` — return list of (min_length, max_length) for each bucket

**`bucket_collate(batch: list[dict]) -> dict`:**
- Same as standard padded collation, but the padding waste should be lower since items in a batch have similar lengths
- Returns `{"ids": list[str], "tokens": (B, max_len) int64, "mask": (B, max_len) int64, "labels": (B,) int64}`
- Pad with 0

**`compute_padding_efficiency(batches: list[dict]) -> dict`:**
- Given a list of collated batches, compute:
  - `"total_tokens"`: sum of all real (non-padding) tokens across all batches
  - `"total_cells"`: sum of batch_size * max_len across all batches (total tensor cells)
  - `"efficiency"`: total_tokens / total_cells (1.0 = no wasted padding)

In [ ]:
class BucketedDataset(Dataset):
    """Dataset that sorts records by length into buckets."""
    # YOUR CODE HERE
    pass


def bucket_collate(batch: list[dict]) -> dict:
    """Collate variable-length sequences with padding."""
    # YOUR CODE HERE
    pass


def compute_padding_efficiency(batches: list[dict]) -> dict:
    """Compute padding efficiency across batches."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 2 ---
torch.manual_seed(42)

# Create records with varying lengths
records = []
for i in range(100):
    length = np.random.randint(5, 50)  # lengths from 5 to 49
    records.append({
        "id": f"item_{i}",
        "tokens": list(np.random.randint(1, 1000, length)),
        "label": np.random.randint(0, 3),
    })

ds = BucketedDataset(records, num_buckets=5)
assert len(ds) == 100

# Check bucket boundaries
boundaries = ds.get_bucket_boundaries()
assert len(boundaries) == 5
# Boundaries should be sorted (buckets go from short to long)
for i in range(len(boundaries) - 1):
    assert boundaries[i][1] <= boundaries[i+1][0] or boundaries[i][1] <= boundaries[i+1][1], \
        f"Buckets should be ordered by length: {boundaries}"

# Single item
item = ds[0]
assert isinstance(item["id"], str)
assert item["tokens"].dtype == torch.int64
assert item["label"].dtype == torch.int64
assert item["label"].ndim == 0  # scalar

# Collation
batch = [ds[i] for i in range(4)]
collated = bucket_collate(batch)
assert collated["tokens"].shape[0] == 4
assert collated["mask"].shape == collated["tokens"].shape
assert collated["labels"].shape == (4,)
assert len(collated["ids"]) == 4

# Mask correctness
for i, item in enumerate(batch):
    orig_len = len(item["tokens"])
    assert collated["mask"][i].sum().item() == orig_len

# Padding efficiency: bucketed should be better than random
# Random batches
random_loader = DataLoader(ds, batch_size=10, collate_fn=bucket_collate, shuffle=True)
random_batches = [b for b in random_loader]

# Sequential (bucketed) batches — items near each other have similar lengths
bucketed_loader = DataLoader(ds, batch_size=10, collate_fn=bucket_collate, shuffle=False)
bucketed_batches = [b for b in bucketed_loader]

random_eff = compute_padding_efficiency(random_batches)
bucketed_eff = compute_padding_efficiency(bucketed_batches)

assert 0 < random_eff["efficiency"] <= 1.0
assert 0 < bucketed_eff["efficiency"] <= 1.0
assert bucketed_eff["efficiency"] >= random_eff["efficiency"] - 0.1, \
    f"Bucketed ({bucketed_eff['efficiency']:.3f}) should be at least as efficient as random ({random_eff['efficiency']:.3f})"

print(f"Random efficiency: {random_eff['efficiency']:.3f}")
print(f"Bucketed efficiency: {bucketed_eff['efficiency']:.3f}")
print("Problem 2: ALL TESTS PASSED")

---

## Problem 3: Multi-Stage Quality Filter

### Scenario

Large-scale training pipelines use hierarchical quality filtering — loose filters for early training, strict for later stages. Build a configurable multi-stage filter.

### Requirements

**`QualityFilter`:**
- `__init__(self, rules: list[dict])` — each rule is a dict with:
  - `"field"`: str — key in the record dict
  - `"op"`: str — one of `"gte"`, `"lte"`, `"gt"`, `"lt"`, `"eq"`, `"in"`, `"not_in"`, `"contains"`
  - `"value"`: the threshold/set to compare against
  - `"required"`: bool (default True) — if the field is missing, does the record fail?
- `apply(self, record: dict) -> bool` — True if record passes ALL rules
- `apply_batch(self, records: list[dict]) -> tuple[list[dict], list[dict]]` — return (passed, failed)
- `report(self, records: list[dict]) -> dict` — return `{"total": int, "passed": int, "failed": int, "failure_reasons": dict}` where failure_reasons maps rule description to count of records that failed that specific rule

**`FilterPipeline`:**
- `__init__(self, stages: list[tuple[str, QualityFilter]])` — named stages applied in order
- `run(self, records: list[dict]) -> dict` — apply each stage sequentially (output of stage N is input to stage N+1)
  - Return `{"final_records": list[dict], "stage_reports": list[dict]}` where each stage report includes the stage name and the QualityFilter report for that stage

In [ ]:
class QualityFilter:
    """Configurable quality filter with rule-based record evaluation."""
    # YOUR CODE HERE
    pass


class FilterPipeline:
    """Multi-stage filter pipeline."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 3 ---

# Basic QualityFilter
qf = QualityFilter([
    {"field": "quality_score", "op": "gte", "value": 0.5},
    {"field": "width", "op": "gte", "value": 256},
    {"field": "format", "op": "in", "value": {"jpg", "png", "webp"}},
])

records = [
    {"id": "a", "quality_score": 0.9, "width": 512, "format": "jpg"},    # pass
    {"id": "b", "quality_score": 0.3, "width": 1024, "format": "png"},   # fail: quality
    {"id": "c", "quality_score": 0.7, "width": 128, "format": "jpg"},    # fail: width
    {"id": "d", "quality_score": 0.8, "width": 512, "format": "bmp"},    # fail: format
    {"id": "e", "quality_score": 0.6, "width": 256, "format": "webp"},   # pass
]

passed, failed = qf.apply_batch(records)
passed_ids = {r["id"] for r in passed}
assert passed_ids == {"a", "e"}, f"Expected {{a, e}}, got {passed_ids}"

# Report
report = qf.report(records)
assert report["total"] == 5
assert report["passed"] == 2
assert report["failed"] == 3
assert len(report["failure_reasons"]) > 0

# Missing field — required=True by default
qf_req = QualityFilter([{"field": "score", "op": "gte", "value": 0.5}])
assert qf_req.apply({"score": 0.8}) == True
assert qf_req.apply({"other": 0.8}) == False  # missing required field

# Missing field — required=False
qf_opt = QualityFilter([{"field": "score", "op": "gte", "value": 0.5, "required": False}])
assert qf_opt.apply({"other": 0.8}) == True  # missing optional field passes

# "contains" operator
qf_contains = QualityFilter([{"field": "caption", "op": "contains", "value": "car"}])
assert qf_contains.apply({"caption": "a red car"}) == True
assert qf_contains.apply({"caption": "a dog"}) == False

# "not_in" operator
qf_notin = QualityFilter([{"field": "source", "op": "not_in", "value": {"spam", "nsfw"}}])
assert qf_notin.apply({"source": "web"}) == True
assert qf_notin.apply({"source": "spam"}) == False

# FilterPipeline
pipeline = FilterPipeline([
    ("coarse", QualityFilter([
        {"field": "quality_score", "op": "gte", "value": 0.3},
    ])),
    ("fine", QualityFilter([
        {"field": "quality_score", "op": "gte", "value": 0.7},
        {"field": "width", "op": "gte", "value": 256},
    ])),
])

all_records = [
    {"id": "1", "quality_score": 0.9, "width": 512},   # survives both
    {"id": "2", "quality_score": 0.1, "width": 1024},  # fails coarse
    {"id": "3", "quality_score": 0.5, "width": 512},   # passes coarse, fails fine (quality)
    {"id": "4", "quality_score": 0.8, "width": 128},   # passes coarse, fails fine (width)
    {"id": "5", "quality_score": 0.75, "width": 256},  # survives both
]

result = pipeline.run(all_records)
final_ids = {r["id"] for r in result["final_records"]}
assert final_ids == {"1", "5"}, f"Expected {{1, 5}}, got {final_ids}"
assert len(result["stage_reports"]) == 2
assert result["stage_reports"][0]["total"] == 5  # coarse sees all 5
assert result["stage_reports"][1]["total"] == 4  # fine sees 4 (after coarse removed 1)

print("Problem 3: ALL TESTS PASSED")

---

## Problem 4: Multi-Head Self-Attention

Implement multi-head self-attention from scratch.

### Formula

```
Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V
```

### Architecture

1. **Single combined projection:** One linear layer `W_qkv` projects input `x` from `d_model` → `3 * d_model` (no bias). Split the output into Q, K, V along the last dimension.
2. **Reshape to heads:** (B, T, d_model) → (B, num_heads, T, d_k) where `d_k = d_model // num_heads`
3. **Scaled dot-product attention** per head using the formula above
4. **Concatenate heads:** (B, num_heads, T, d_k) → (B, T, d_model)
5. **Output projection:** `W_out` linear layer `d_model → d_model` (no bias)

### Causal Mask

When `causal=True`, set attention scores where `query_pos < key_pos` to `-inf` before softmax. This prevents each position from attending to future positions.

### Constraints

- Assert `d_model % num_heads == 0` in `__init__`
- Do NOT use `nn.MultiheadAttention` or `F.scaled_dot_product_attention`
- No bias in any linear layers

### Interface

```python
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor, causal: bool = False) -> torch.Tensor:
        # x: (B, T, d_model) -> output: (B, T, d_model)
```

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, x: torch.Tensor, causal: bool = False) -> torch.Tensor:
        # YOUR CODE HERE
        pass

In [ ]:
# --- Tests: Problem 4 ---
torch.manual_seed(42)

mha = MultiHeadSelfAttention(d_model=64, num_heads=8)
x = torch.randn(2, 10, 64)

# Shape preservation
out = mha(x)
assert out.shape == (2, 10, 64), f"Expected (2,10,64), got {out.shape}"

# Causal forward
out_causal = mha(x, causal=True)
assert out_causal.shape == (2, 10, 64)

# Causal property: output at position t must not depend on positions > t
x_test = torch.randn(1, 5, 64)
out1 = mha(x_test, causal=True)
x_modified = x_test.clone()
x_modified[0, 3:, :] = torch.randn(2, 64)  # Change positions 3 and 4
out2 = mha(x_modified, causal=True)
assert torch.allclose(out1[0, :3], out2[0, :3], atol=1e-5), "Causal mask broken: early positions changed"
assert not torch.allclose(out1[0, 3], out2[0, 3], atol=1e-3), "Position 3 should differ"

# Parameter count: W_qkv (d * 3d, no bias) + W_out (d * d, no bias)
total_params = sum(p.numel() for p in mha.parameters())
assert total_params == 64 * 64 * 3 + 64 * 64, f"Unexpected param count: {total_params}"

# Gradient flow
x_grad = torch.randn(2, 10, 64, requires_grad=True)
out_grad = mha(x_grad)
out_grad.sum().backward()
assert x_grad.grad is not None, "Gradient should flow to input"
for name, p in mha.named_parameters():
    assert p.grad is not None, f"No gradient for {name}"

# Invalid config
try:
    MultiHeadSelfAttention(d_model=63, num_heads=8)
    assert False, "Should have raised error for d_model not divisible by num_heads"
except (AssertionError, ValueError):
    pass

# Single head should work
mha_single = MultiHeadSelfAttention(d_model=32, num_heads=1)
assert mha_single(torch.randn(1, 5, 32)).shape == (1, 5, 32)

# Batch size 1
assert mha(torch.randn(1, 3, 64)).shape == (1, 3, 64)

print("Problem 4: ALL TESTS PASSED")

---

## Problem 5: Gradient Descent Variants (SGD, Momentum, Adam)

**Scenario:** Implement three optimizer variants from scratch. This tests your understanding of the mechanics behind `torch.optim`.

**Spec:**

All optimizers implement:
- `__init__(self, params: list[torch.Tensor], lr: float, **kwargs)` — store references to parameter tensors
- `step(self)` — update parameters in-place
- `zero_grad(self)` — zero all parameter gradients

**SGD with Momentum:**
$$v_t = \beta \cdot v_{t-1} + g_t$$
$$\theta_t = \theta_{t-1} - \eta \cdot v_t$$

Initialize velocity $v_0 = 0$. Default $\beta = 0.9$.

**Adam:**
$$m_t = \beta_1 \cdot m_{t-1} + (1 - \beta_1) \cdot g_t$$
$$v_t = \beta_2 \cdot v_{t-1} + (1 - \beta_2) \cdot g_t^2$$
$$\hat{m}_t = m_t / (1 - \beta_1^t)$$
$$\hat{v}_t = v_t / (1 - \beta_2^t)$$
$$\theta_t = \theta_{t-1} - \eta \cdot \hat{m}_t / (\sqrt{\hat{v}_t} + \epsilon)$$

Initialize $m_0 = 0$, $v_0 = 0$, $t = 0$ (increment at each step). Default $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$.

**Constraints:**
- No `torch.optim`. Modify tensors with `param.data -= ...` (in-place on .data to avoid autograd tracking).
- Each step() should work on all params in the list.

In [ ]:
class SGDMomentum:
    """SGD with momentum optimizer.

    Args:
        params: List of parameter tensors.
        lr: Learning rate.
        momentum: Momentum coefficient (default 0.9).
    """

    def __init__(self, params: list[torch.Tensor], lr: float, momentum: float = 0.9):
        # YOUR CODE HERE
        pass

    def step(self):
        # YOUR CODE HERE
        pass

    def zero_grad(self):
        # YOUR CODE HERE
        pass


class Adam:
    """Adam optimizer.

    Args:
        params: List of parameter tensors.
        lr: Learning rate.
        betas: Tuple of (beta1, beta2).
        eps: Epsilon for numerical stability.
    """

    def __init__(self, params: list[torch.Tensor], lr: float,
                 betas: tuple[float, float] = (0.9, 0.999), eps: float = 1e-8):
        # YOUR CODE HERE
        pass

    def step(self):
        # YOUR CODE HERE
        pass

    def zero_grad(self):
        # YOUR CODE HERE
        pass

In [ ]:
# --- Tests: Problem 5 ---
torch.manual_seed(42)

# Test SGD with Momentum against torch.optim.SGD
w1 = torch.randn(10, 5, requires_grad=True)
w2 = w1.data.clone().requires_grad_(True)

opt_custom = SGDMomentum([w1], lr=0.01, momentum=0.9)
opt_torch = torch.optim.SGD([w2], lr=0.01, momentum=0.9)

for _ in range(20):
    loss1 = (w1 ** 2).sum()
    loss1.backward()
    opt_custom.step()
    opt_custom.zero_grad()

    loss2 = (w2 ** 2).sum()
    loss2.backward()
    opt_torch.step()
    opt_torch.zero_grad()

assert torch.allclose(w1.data, w2.data, atol=1e-5), \
    f"SGD Momentum should match torch.optim.SGD. Max diff: {(w1.data - w2.data).abs().max():.6f}"

# Test Adam against torch.optim.Adam
a1 = torch.randn(8, 4, requires_grad=True)
a2 = a1.data.clone().requires_grad_(True)

opt_adam_custom = Adam([a1], lr=0.001, betas=(0.9, 0.999))
opt_adam_torch = torch.optim.Adam([a2], lr=0.001, betas=(0.9, 0.999), eps=1e-8)

for _ in range(50):
    loss1 = (a1 ** 2).sum()
    loss1.backward()
    opt_adam_custom.step()
    opt_adam_custom.zero_grad()

    loss2 = (a2 ** 2).sum()
    loss2.backward()
    opt_adam_torch.step()
    opt_adam_torch.zero_grad()

assert torch.allclose(a1.data, a2.data, atol=1e-4), \
    f"Adam should match torch.optim.Adam. Max diff: {(a1.data - a2.data).abs().max():.6f}"

# zero_grad should zero all grads
p = torch.randn(5, requires_grad=True)
(p ** 2).sum().backward()
assert p.grad is not None and p.grad.abs().sum() > 0
opt_test = SGDMomentum([p], lr=0.01)
opt_test.zero_grad()
assert (p.grad == 0).all(), "zero_grad should zero gradients"

# Optimizers should reduce loss
w_opt = torch.randn(20, requires_grad=True)
opt = Adam([w_opt], lr=0.01)
initial_loss = (w_opt ** 2).sum().item()
for _ in range(100):
    loss = (w_opt ** 2).sum()
    loss.backward()
    opt.step()
    opt.zero_grad()
final_loss = (w_opt ** 2).sum().item()
assert final_loss < initial_loss * 0.1, f"Adam should significantly reduce loss: {initial_loss:.2f} -> {final_loss:.2f}"

# Multiple parameters
p1 = torch.randn(3, requires_grad=True)
p2 = torch.randn(4, requires_grad=True)
opt_multi = Adam([p1, p2], lr=0.01)
loss = (p1 ** 2).sum() + (p2 ** 2).sum()
loss.backward()
opt_multi.step()
opt_multi.zero_grad()
assert p1.grad is not None and (p1.grad == 0).all()
assert p2.grad is not None and (p2.grad == 0).all()

print("Problem 5: ALL TESTS PASSED")

---

## Problem 6: Neural Network + Manual Backprop (No Autograd)

**Scenario:** Prove you understand what autograd does under the hood. Implement a 2-layer neural network with forward pass, loss, and backward pass entirely from scratch.

**Spec:**

Build a 2-layer MLP: Input → Linear → ReLU → Linear → Softmax Cross-Entropy Loss

**Architecture:**
- Layer 1: $h = \text{ReLU}(X W_1 + b_1)$ where $W_1$: (D, H), $b_1$: (H,)
- Layer 2: $s = h W_2 + b_2$ where $W_2$: (H, C), $b_2$: (C,) — raw scores (logits)

**Loss:** Softmax + Cross-Entropy
$$p = \text{softmax}(s)$$
$$L = -\frac{1}{N} \sum_i \log(p_{i, y_i})$$

**Backward pass gradients (all provided):**

$$\frac{\partial L}{\partial s} = \frac{1}{N}(p - \text{one\_hot}(y))$$

$$\frac{\partial L}{\partial W_2} = h^T \cdot \frac{\partial L}{\partial s}$$

$$\frac{\partial L}{\partial b_2} = \sum_{\text{batch}} \frac{\partial L}{\partial s}$$

$$\frac{\partial L}{\partial h} = \frac{\partial L}{\partial s} \cdot W_2^T$$

$$\frac{\partial L}{\partial h_{pre}} = \frac{\partial L}{\partial h} \odot \mathbb{1}[h_{pre} > 0]$$ (ReLU gradient)

$$\frac{\partial L}{\partial W_1} = X^T \cdot \frac{\partial L}{\partial h_{pre}}$$

$$\frac{\partial L}{\partial b_1} = \sum_{\text{batch}} \frac{\partial L}{\partial h_{pre}}$$

**`ManualMLP`:**
- `__init__(self, input_dim, hidden_dim, num_classes, lr=0.01)` — initialize weights with `torch.randn(...) * 0.01`, biases with zeros
- `forward(self, X)` — return (loss, scores) given X: (N, D) and stored labels
- `train_step(self, X, y)` — full forward + backward + weight update. Return loss (float).
- `predict(self, X)` — return (N,) int64 predicted class labels

**Constraints:**
- NO autograd (no .backward(), no requires_grad). All gradients computed manually.
- Use only basic tensor operations.

In [ ]:
class ManualMLP:
    """2-layer MLP with manual backpropagation.

    Args:
        input_dim: Number of input features.
        hidden_dim: Number of hidden units.
        num_classes: Number of output classes.
        lr: Learning rate.
    """

    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int, lr: float = 0.01):
        # YOUR CODE HERE
        pass

    def forward(self, X: torch.Tensor, y: torch.Tensor) -> tuple[float, torch.Tensor]:
        """Forward pass. Returns (loss, scores)."""
        # YOUR CODE HERE
        pass

    def train_step(self, X: torch.Tensor, y: torch.Tensor) -> float:
        """Forward + backward + update. Returns loss."""
        # YOUR CODE HERE
        pass

    def predict(self, X: torch.Tensor) -> torch.Tensor:
        """Predict class labels."""
        # YOUR CODE HERE
        pass

In [ ]:
# --- Tests: Problem 6 ---
torch.manual_seed(42)

# Create linearly separable data
N, D, H, C = 200, 10, 32, 3
X = torch.randn(N, D)
# Create labels based on first feature
y = torch.zeros(N, dtype=torch.int64)
y[X[:, 0] > 0.5] = 1
y[X[:, 0] < -0.5] = 2

mlp = ManualMLP(input_dim=D, hidden_dim=H, num_classes=C, lr=0.1)

# Initial predictions should be random-ish
preds_init = mlp.predict(X)
assert preds_init.shape == (N,)
assert preds_init.dtype == torch.int64
init_acc = (preds_init == y).float().mean().item()
# Random would be ~33%, shouldn't be much better before training
assert init_acc < 0.6, f"Initial accuracy should be low, got {init_acc:.1%}"

# Train and verify loss decreases
losses = []
for _ in range(200):
    loss = mlp.train_step(X, y)
    losses.append(loss)

assert losses[-1] < losses[0], f"Loss should decrease: {losses[0]:.4f} -> {losses[-1]:.4f}"

# After training, accuracy should be decent
preds_final = mlp.predict(X)
final_acc = (preds_final == y).float().mean().item()
assert final_acc > 0.7, f"Accuracy after training should be > 70%, got {final_acc:.1%}"

# Verify forward returns reasonable loss
loss_val, scores = mlp.forward(X, y)
assert isinstance(loss_val, float)
assert loss_val > 0
assert scores.shape == (N, C)

# Verify no autograd is used (weights should not require grad)
assert not mlp.W1.requires_grad, "W1 should not require grad (manual backprop)"
assert not mlp.W2.requires_grad, "W2 should not require grad (manual backprop)"

# Gradient check: compare manual gradients to autograd (sanity check)
mlp_check = ManualMLP(input_dim=4, hidden_dim=8, num_classes=3, lr=0.01)
X_check = torch.randn(10, 4)
y_check = torch.randint(0, 3, (10,))

# Get manual gradients by doing a train_step but saving grads before update
# We'll just verify training reduces loss (gradient direction is correct)
loss_before = mlp_check.forward(X_check, y_check)[0]
mlp_check.train_step(X_check, y_check)
loss_after = mlp_check.forward(X_check, y_check)[0]
assert loss_after < loss_before, "Single step should reduce loss (correct gradient direction)"

print("Problem 6: ALL TESTS PASSED")

---

## Scoring

| Result | Assessment |
|--------|------------|
| Solved in < 25 min, all tests pass | **Strong pass** |
| Solved in 25-30 min, minor debugging | **Pass** — exam-ready |
| 30-40 min or needed hints | **Borderline** — redo tomorrow |
| Couldn't solve or > 40 min | **Go back to MEDIUM** — nail those first |